In [2]:
import subprocess
from threading import Thread
from tqdm import tqdm

def run_shell_script(script_path, args_list):
    # Create a list to store the subprocesses
    processes = []

    # Create a list to store the progress bars
    progress_bars = []

    # Iterate over the arguments list
    for i, args in enumerate(args_list):
        # Create a subprocess for each set of arguments
        process = subprocess.Popen([script_path] + args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        # Create a progress bar for each subprocess
        progress_bar = tqdm(total=100, desc=f"Process {i+1}", unit="%", bar_format="{desc}: {percentage:.0f}%|{bar}|")

        # Append the subprocess and progress bar to their respective lists
        processes.append(process)
        progress_bars.append(progress_bar)

        # Define a function to update the progress bar
        def update_progress(process, progress_bar):
            while process.poll() is None:
                # Read the output of the subprocess
                output = process.stdout.readline().decode().strip()

                # Update the progress bar based on the output
                if output:
                    progress = int(output)
                    progress_bar.update(progress - progress_bar.n)

        # Create a thread to update the progress bar
        thread = Thread(target=update_progress, args=(process, progress_bar))
        thread.start()

    # Wait for all subprocesses to complete
    for process in processes:
        process.wait()

    # Close all progress bars
    for progress_bar in progress_bars:
        progress_bar.close()
